In [1]:
# automatically reload imported modules before executing code

%load_ext autoreload
%autoreload 2

In [2]:
from pyrekordbox import Rekordbox6Database
import polars as pl
from nbutils import setup_path

setup_path()
db = Rekordbox6Database()

pl.Config.set_tbl_rows(20)  # Show 100 row

[14:06:16] pyrekordbox.db6.database:WARNING  - Rekordbox is running!


polars.config.Config

In [3]:
from utils import get_base_dataset

base_df = get_base_dataset(db, min_tag_count=5)


Filtering tags with fewer than 5 occurrences:

Genre:
  - Grime: 1 occurrence(s)
  - Dancehall: 3 occurrence(s)
  - New Beat: 3 occurrence(s)
  - Gabber: 4 occurrence(s)
  - Blues: 4 occurrence(s)

Mood:
  - Industrial: 1 occurrence(s)

Total tags filtered: 6



In [16]:
from sklearn.model_selection import train_test_split

from processing import (
    prepare_multilabel_data, 
    PolarsStandardScaler, 
    PolarsPCA,
    get_multilabel_stats,
    filter_rare_labels
)

# Load features
features_df = pl.read_parquet("../data/song_features.parquet")

# Define feature columns to exclude
exclude_cols = [
    "song_path",
    "harmonic_percussive_ratio",
    "percussive_strength",
    "tonnetz_mean_0", "tonnetz_mean_1", "tonnetz_mean_2", "tonnetz_mean_3", "tonnetz_mean_4", "tonnetz_mean_5",
    "tonnetz_std_0", "tonnetz_std_1", "tonnetz_std_2", "tonnetz_std_3", "tonnetz_std_4", "tonnetz_std_5"
]

# Get feature columns
feature_cols = [col for col in features_df.columns 
                if col not in exclude_cols + ["song_id", "song_path"]]

# Filter features to only include non-null energy_increase_ratio
features_df = features_df.filter(pl.col("energy_increase_ratio").is_not_null())

# Configuration
test_size = 0.2
random_state = 42
min_train_count = 10
pca_variance = 0.95


def preprocess_tag_group(
    base_df, 
    features_df, 
    tag_group, 
    feature_cols, 
    test_size, 
    random_state, 
    min_train_count, 
    pca_variance
):
    """
    Complete preprocessing pipeline for one tag group.
    
    Returns:
    -------
    X_train, X_test : pl.DataFrame
        Preprocessed feature matrices
    y_train, y_test : pl.DataFrame
        Binary label matrices
    tags : list[str]
        List of tag names after filtering
    song_ids_train, song_ids_test : pl.DataFrame
        Song IDs for train and test sets (for joining with metadata)
    scaler : PolarsStandardScaler
        Fitted scaler (for future inference)
    pca : PolarsPCA
        Fitted PCA transformer (for future inference)
    """
    print(f"\n{'='*70}")
    print(f"PREPROCESSING: {tag_group.upper()}")
    print(f"{'='*70}")
    
    # 1. Prepare multi-label data
    X, y, tags = prepare_multilabel_data(
        base_df=base_df,
        features_df=features_df,
        tag_group=tag_group,
        feature_columns=feature_cols
    )
    
    # Show initial label distribution
    print(f"\n{tag_group} - Initial label statistics:")
    print(get_multilabel_stats(y, tags))
    
    # IMPORTANT: Save song IDs before dropping them
    song_ids = X.select("song_id")
    X_features = X.drop("song_id")
    
    # 2. Split train/test at SONG level
    # Split features, labels, AND song IDs with the same random state
    X_train, X_test, y_train, y_test, song_ids_train, song_ids_test = train_test_split(
        X_features, 
        y,
        song_ids,
        test_size=test_size,
        random_state=random_state
    )
    
    # 3. Filter rare labels based on training set
    X_train, y_train, X_test, y_test, tags = filter_rare_labels(
        X_train, y_train, 
        X_test, y_test, 
        tags, 
        min_count=min_train_count
    )
    
    # 4. Normalize features (fit on training data only)
    print(f"Normalizing features...")
    scaler = PolarsStandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 5. Apply PCA (fit on training data only)
    pca = PolarsPCA(n_components=pca_variance)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)
    
    # Summary
    print(f"\n{tag_group} - Final dataset summary:")
    print(f"  Original features: {X_train_scaled.shape[1]}")
    print(f"  PCA components: {X_train_pca.shape[1]}")
    print(f"  Variance preserved: {pca.explained_variance_ratio_.sum():.2%}")
    print(f"  Train samples: {X_train_pca.shape[0]}")
    print(f"  Test samples: {X_test_pca.shape[0]}")
    print(f"  Active labels: {len(tags)}")
    print(f"  Avg tags/song (train): {y_train.select(pl.sum_horizontal(pl.all()).mean()).item():.2f}")
    print(f"{'='*70}\n")
    
    # Return song IDs as well!
    return X_train_pca, X_test_pca, y_train, y_test, tags, song_ids_train, song_ids_test, scaler, pca


# ============================================================================
# PROCESS ALL TAG GROUPS
# ============================================================================

# MOOD
(X_mood_train, X_mood_test, y_mood_train, y_mood_test, mood_tags, 
 mood_ids_train, mood_ids_test, scaler_mood, pca_mood) = \
    preprocess_tag_group(
        base_df, features_df, "Mood", feature_cols,
        test_size, random_state, min_train_count, pca_variance
    )

# SITUATION
(X_situation_train, X_situation_test, y_situation_train, y_situation_test, situation_tags,
 situation_ids_train, situation_ids_test, scaler_situation, pca_situation) = \
    preprocess_tag_group(
        base_df, features_df, "Situation", feature_cols,
        test_size, random_state, min_train_count, pca_variance
    )

# GENRE
(X_genre_train, X_genre_test, y_genre_train, y_genre_test, genre_tags,
 genre_ids_train, genre_ids_test, scaler_genre, pca_genre) = \
    preprocess_tag_group(
        base_df, features_df, "Genre", feature_cols,
        test_size, random_state, min_train_count, pca_variance
    )

print("\n" + "="*70)
print("PREPROCESSING COMPLETE")
print("="*70)
print(f"Ready to train models for {len(mood_tags) + len(situation_tags) + len(genre_tags)} total labels")
print("="*70 + "\n")


PREPROCESSING: MOOD

Preparing multi-label data for 'Mood':
Unique songs: 686
Unique tags: 23
Tags: Chill, Dancefloor, Dark, Deep, Experimental, Good Vibes, Groovy, Guilty, Hippy, Like a boss, Love, Minimal, Mysterious, Pretty, Sad, Spacy, Stadium, Sunset, Synths, Temposhifter, Trippy, Uplifting, Uptempo

Final dataset shape:
  X: (496, 361) (song_id + 360 features)
  y: (496, 23) (23 binary labels)
  Average tags per song: 4.33


Mood - Initial label statistics:
shape: (23, 3)
┌──────────────┬───────┬────────────┐
│ tag          ┆ count ┆ percentage │
│ ---          ┆ ---   ┆ ---        │
│ str          ┆ i64   ┆ f64        │
╞══════════════╪═══════╪════════════╡
│ Good Vibes   ┆ 281   ┆ 56.653226  │
│ Dancefloor   ┆ 278   ┆ 56.048387  │
│ Uplifting    ┆ 259   ┆ 52.217742  │
│ Sunset       ┆ 231   ┆ 46.572581  │
│ Groovy       ┆ 211   ┆ 42.540323  │
│ Deep         ┆ 133   ┆ 26.814516  │
│ Chill        ┆ 123   ┆ 24.798387  │
│ Pretty       ┆ 112   ┆ 22.580645  │
│ Minimal      ┆ 79   

In [17]:
from models import (
    get_linear_model,
    get_knn_model,
    get_decision_tree_model,
    get_random_forest_model,
    get_xgboost_model,
    evaluate_model,
    compare_models,
    predictions_to_labels,
    predictions_to_dataframe,
)

# ============================================================================
# OPTION 1: Train and evaluate a single model
# ============================================================================
model = get_random_forest_model(n_estimators=100, max_depth=15)
model.fit(X_genre_train.to_numpy(), y_genre_train.to_numpy())

# Evaluate with detailed metrics
metrics = evaluate_model(model, X_genre_test, y_genre_test, genre_tags)

# Get predictions as labels
y_pred = metrics['predictions']
predicted_labels = predictions_to_labels(y_pred, genre_tags)

print("First 5 predictions:")
for i, labels in enumerate(predicted_labels[:5]):
    print(f"  Song {i+1}: {labels}")


MODEL EVALUATION RESULTS
Hamming Loss: 0.0937
  (Average fraction of labels incorrectly predicted per sample)

Exact Match Accuracy: 0.0100
  (Fraction of samples with ALL labels predicted correctly)

Macro-Averaged Metrics:
  Precision: 0.1121
  Recall:    0.0436
  F1-Score:  0.0485

Per-Label Metrics (sorted by F1-score):
shape: (30, 5)
┌─────────────┬───────────┬──────────┬──────────┬─────────┐
│ tag         ┆ precision ┆ recall   ┆ f1_score ┆ support │
│ ---         ┆ ---       ┆ ---      ┆ ---      ┆ ---     │
│ str         ┆ f64       ┆ f64      ┆ f64      ┆ i64     │
╞═════════════╪═══════════╪══════════╪══════════╪═════════╡
│ Soul        ┆ 0.741379  ┆ 0.754386 ┆ 0.747826 ┆ 57      │
│ House       ┆ 0.62069   ┆ 0.461538 ┆ 0.529412 ┆ 39      │
│ Disco       ┆ 1.0       ┆ 0.047619 ┆ 0.090909 ┆ 21      │
│ Funk        ┆ 1.0       ┆ 0.045455 ┆ 0.086957 ┆ 22      │
│ Ambient     ┆ 0.0       ┆ 0.0      ┆ 0.0      ┆ 17      │
│ Bass        ┆ 0.0       ┆ 0.0      ┆ 0.0      ┆ 10      

In [18]:
metrics

{'hamming_loss': 0.09366666666666666,
 'exact_match_accuracy': 0.01,
 'macro_precision': 0.11206896551724137,
 'macro_recall': 0.043633267317477846,
 'macro_f1': 0.04850344881035418,
 'per_label_metrics': shape: (30, 5)
 ┌─────────────┬───────────┬──────────┬──────────┬─────────┐
 │ tag         ┆ precision ┆ recall   ┆ f1_score ┆ support │
 │ ---         ┆ ---       ┆ ---      ┆ ---      ┆ ---     │
 │ str         ┆ f64       ┆ f64      ┆ f64      ┆ i64     │
 ╞═════════════╪═══════════╪══════════╪══════════╪═════════╡
 │ Soul        ┆ 0.741379  ┆ 0.754386 ┆ 0.747826 ┆ 57      │
 │ House       ┆ 0.62069   ┆ 0.461538 ┆ 0.529412 ┆ 39      │
 │ Disco       ┆ 1.0       ┆ 0.047619 ┆ 0.090909 ┆ 21      │
 │ Funk        ┆ 1.0       ┆ 0.045455 ┆ 0.086957 ┆ 22      │
 │ Ambient     ┆ 0.0       ┆ 0.0      ┆ 0.0      ┆ 17      │
 │ Bass        ┆ 0.0       ┆ 0.0      ┆ 0.0      ┆ 10      │
 │ Beats       ┆ 0.0       ┆ 0.0      ┆ 0.0      ┆ 10      │
 │ Boogie      ┆ 0.0       ┆ 0.0      ┆ 0.0     

In [19]:
# ============================================================================
# OPTION 2: Compare multiple models at once
# ============================================================================
models = {
    "Linear": get_linear_model(),
    "KNN": get_knn_model(n_neighbors=10),
    "Decision Tree": get_decision_tree_model(max_depth=15),
    "Random Forest": get_random_forest_model(n_estimators=100),
    "XGBoost": get_xgboost_model(n_estimators=100),
}

# This will train all models and compare them
comparison = compare_models(
    models, 
    X_genre_train, y_genre_train,
    X_genre_test, y_genre_test,
    genre_tags
)

# Results sorted by F1-score
print(comparison)


Training: Linear
 Linear trained successfully
  Hamming Loss: 0.1067, Exact Match: 0.0100, F1: 0.2150

Training: KNN
 KNN trained successfully
  Hamming Loss: 0.0990, Exact Match: 0.0100, F1: 0.1108

Training: Decision Tree
 Decision Tree trained successfully
  Hamming Loss: 0.1307, Exact Match: 0.0000, F1: 0.1546

Training: Random Forest
 Random Forest trained successfully
  Hamming Loss: 0.0937, Exact Match: 0.0100, F1: 0.0485

Training: XGBoost
 XGBoost trained successfully
  Hamming Loss: 0.0970, Exact Match: 0.0200, F1: 0.1205

MODEL COMPARISON SUMMARY
shape: (5, 6)
┌───────────────┬──────────────┬──────────────────────┬─────────────────┬──────────────┬──────────┐
│ model         ┆ hamming_loss ┆ exact_match_accuracy ┆ macro_precision ┆ macro_recall ┆ macro_f1 │
│ ---           ┆ ---          ┆ ---                  ┆ ---             ┆ ---          ┆ ---      │
│ str           ┆ f64          ┆ f64                  ┆ f64             ┆ f64          ┆ f64      │
╞═══════════════

In [22]:
from utils import get_clean_songs

songs_df = get_clean_songs(db, rename=True)


songs_df

song_id,song_path,song_title,artist_id,artist_name,genre_id,genre_name,bpm,date_created,length,tag_names,tag_ids,sample_rate,has_tags
str,str,str,str,str,str,str,i32,str,i32,list[str],list[str],i32,bool
"""36085625""","""/Users/quintenrosseel/Music/Pi…","""Surrender""","""1673664596""","""Gerd Janson""","""159652946""","""Techno/House""",12200,"""2021-12-26""",413,"[""AUTOTAG""]","[""2966753611""]",44100,false
"""18988943""","""/Users/quintenrosseel/Music/Pi…","""I Want Your Soul""","""1642798025""","""Armand Van Helden""","""3027895697""","""Classics/House""",12800,"""2021-12-26""",399,"[""AUTOTAG""]","[""2966753611""]",44100,false
"""118919255""","""/Users/quintenrosseel/Music/Pi…","""Save Our Love""","""2508199194""","""Escape From New York""","""3584637391""","""Disco""",11010,"""2022-08-23""",304,"[""80s"", ""Disco"", … ""Chill""]","[""1155275430"", ""62609778"", … ""3718426834""]",44100,true
"""228400738""","""/Users/quintenrosseel/Music/Pi…","""RENT4""","""3953188055""","""Lakim""","""4282047218""","""Garage""",14500,"""2021-07-17""",167,"[""AUTOTAG""]","[""2966753611""]",44100,false
"""217615930""","""/Users/quintenrosseel/Music/Pi…","""Funky Child (1993)""","""1137483595""","""Lords Of The Underground""","""2864790501""","""Hip-hop""",9630,"""2018-10-11""",227,"[""Beats"", ""Hip-Hop"", … ""TAG YEAR""]","[""2053683127"", ""3917148722"", … ""382831235""]",44100,true
"""121941058""","""/Users/quintenrosseel/Music/Pi…","""I Wanna Do The Do (Extended Re…","""3249033290""","""Bobby Rush""","""1520704553""","""Funk/Soul/Motown/Jazz""",10990,"""2018-10-11""",390,"[""AUTOTAG""]","[""2966753611""]",44100,false
"""261795999""","""/Users/quintenrosseel/Music/Pi…","""Days Like This (DJ Spinna Remi…","""645442704""","""Shaun Escoffery""","""169675979""","""Disco/House""",11990,"""2018-10-11""",486,"[""AUTOTAG""]","[""2966753611""]",44100,false
"""191218482""","""/Users/quintenrosseel/Music/Pi…","""Dancer""","""276731648""","""Gino Soccio""","""3584637391""","""Disco""",12190,"""2019-01-11""",507,"[""AUTOTAG""]","[""2966753611""]",44100,false
"""55055504""","""/Users/quintenrosseel/Music/Pi…","""Untitled 02""","""4081332306""","""Unknown Artist""","""1935845763""","""Warmup/House""",12300,"""2018-10-11""",475,"[""Burning Man"", ""Loungy"", … ""Trippy""]","[""169548333"", ""4144163695"", … ""938485439""]",44100,true


In [26]:
from models import get_random_forest_model, predictions_to_dataframe

# Train model
best_model = get_random_forest_model(n_estimators=200, max_depth=20)
best_model.fit(X_genre_train.to_numpy(), y_genre_train.to_numpy())

# Predict
y_pred = best_model.predict(X_genre_test.to_numpy())

# Get test song IDs
test_song_ids = genre_ids_test["song_id"].to_list()

# Convert predictions to DataFrame
predictions_df = predictions_to_dataframe(y_pred, genre_tags, test_song_ids)

# Join with songs_df to get artist and title
predictions_with_metadata = predictions_df.join(
    songs_df.select(["song_id", "artist_name", "song_title"]),
    left_on="song_id",
    right_on="song_id",
    how="left"
).select(["song_id", "artist_name", "song_title", "predicted_tags"])

print("Predictions with Song Metadata:")
print(predictions_with_metadata)

Predictions with Song Metadata:
shape: (100, 4)
┌───────────┬─────────────────────────┬─────────────────────────────────┬────────────────┐
│ song_id   ┆ artist_name             ┆ song_title                      ┆ predicted_tags │
│ ---       ┆ ---                     ┆ ---                             ┆ ---            │
│ str       ┆ str                     ┆ str                             ┆ list[str]      │
╞═══════════╪═════════════════════════╪═════════════════════════════════╪════════════════╡
│ 79369571  ┆ Bill Withers            ┆ Lovely Day                      ┆ ["Soul"]       │
│ 135893281 ┆ Mirage                  ┆ Summer Grooves                  ┆ ["Soul"]       │
│ 213251833 ┆ Masarima                ┆ Freak Like U (Club Mix)         ┆ []             │
│ 181265614 ┆ Phyllis Hyman           ┆ You Know How to Love Me (Long … ┆ ["Soul"]       │
│ 216336789 ┆ Pacific Coliseum        ┆ Ocean City                      ┆ ["House"]      │
│ 62938980  ┆ The Just Brothers       ┆ Sl

In [31]:
predictions_with_metadata.filter(
    pl.col('predicted_tags').list.len() > 1
)

song_id,artist_name,song_title,predicted_tags
str,str,str,list[str]
"""191048045""","""Peven Everet""","""Stuck""","[""House"", ""Soul""]"
"""149005306""","""Soul Providers""","""Rise (Bini & Martini Original …","[""House"", ""Soul""]"
"""113855269""","""Bailey Ibbs""","""What's My Chance?""","[""House"", ""Soul""]"
"""166189664""","""Honey Dijon""","""Work (feat. Dave Giles II, Cor…","[""House"", ""Soul""]"
"""71433424""","""Lio""","""Sage comme une image (Long Ver…","[""House"", ""Soul""]"
"""238045823""","""Tim Deluxe""","""JAS (Club Mix)""","[""House"", ""Soul""]"
"""20679104""","""Supershy feat Wayne Snow""","""Change""","[""House"", ""Soul""]"
"""119664866""","""SNAP!""","""Rhythm Is a Dancer""","[""House"", ""Soul""]"
"""158755307""","""Skip Mahoney""","""Janice (Don't Be So Blind To L…","[""Funk"", ""Soul""]"


In [33]:
# Explode the list column and get unique values
unique_predicted_tags = (
    predictions_with_metadata
    .select("predicted_tags")
    .explode("predicted_tags")
    .unique()
    .sort("predicted_tags")
)

print("Unique predicted tags:")
print(unique_predicted_tags)

Unique predicted tags:
shape: (4, 1)
┌────────────────┐
│ predicted_tags │
│ ---            │
│ str            │
╞════════════════╡
│ null           │
│ Funk           │
│ House          │
│ Soul           │
└────────────────┘
